# Task 2: Content Type Prediction (Movie vs TV Show)

Binary classification predicting a title's format from its metadata.

- **Target:** $y \in \{\text{Movie}, \text{TV Show}\}$
- **Features:** format-neutral genres, audience rating, release year

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.data_loader import get_preprocessed_data

df = get_preprocessed_data('../data/Dataset.csv')
print(f"Loaded {len(df):,} titles with {len(df.columns)} columns.")
print(df['type'].value_counts(normalize=True).round(3))

Loaded 8,790 titles with 19 columns.
type
Movie      0.697
TV Show    0.303
Name: proportion, dtype: float64


## 1. Avoiding Target Leakage

The catalog's raw genre labels name the format ("TV Dramas" vs "Dramas", "Docuseries" vs
"Documentaries"). Every raw label maps to only one format, so a model trained on them simply reads the
answer and reaches a meaningless ~100% accuracy.

The pipeline therefore maps each label to a **format-neutral genre** before training.

In [2]:
leak = (df.assign(genre=df['listed_in'].str.split(', ')).explode('genre')
        .groupby('genre')['type'].agg(lambda s: (s == 'Movie').mean()))
print(f"Raw genre labels that map to a single format: {((leak == 0) | (leak == 1)).mean():.0%}")
df[['listed_in', 'genres_neutral']].drop_duplicates().head(8)

Raw genre labels that map to a single format: 100%


,listed_in,genres_neutral
0,Documentaries,Documentary
1,"Crime TV Shows, International TV Shows, TV Act...","Crime, International, Action & Adventure"
2,"TV Dramas, TV Horror, TV Mysteries","Drama, Horror, Mystery"
3,"Children & Family Movies, Comedies","Kids & Family, Comedy"
4,"Dramas, Independent Movies, International Movies","Drama, Independent, International"
5,"British TV Shows, Reality TV","British, Reality"
6,"Comedies, Dramas","Comedy, Drama"
7,"Children & Family Movies, Comedies, Music & Mu...","Kids & Family, Comedy, Music & Musicals"


## 2. Model Training & Comparison

In [3]:
from src.classifier import ContentTypeClassifier

rf_model = ContentTypeClassifier(model_type='rf')
rf_results = rf_model.train(df)
lr_model = ContentTypeClassifier(model_type='lr')
lr_results = lr_model.train(df)

pd.DataFrame([
    {"Model": name, **{k: r[k] for k in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']}}
    for name, r in [("Random Forest", rf_results), ("Logistic Regression", lr_results)]
])

,Model,accuracy,precision,recall,f1_score,roc_auc
0,Random Forest,0.8441,0.8614,0.8441,0.8482,0.9437
1,Logistic Regression,0.8441,0.8491,0.8441,0.8459,0.9242


## 3. Confusion Matrix (Random Forest)

In [4]:
classes = rf_results['classes']
pd.DataFrame(rf_results['confusion_matrix'],
             index=[f"Actual {c}" for c in classes],
             columns=[f"Pred {c}" for c in classes])

,Pred Movie,Pred TV Show
Actual Movie,1024,201
Actual TV Show,73,460


## 4. Inference Demo

In [5]:
test_cases = [
    {"listed_in": "Documentaries, International Movies", "rating": "PG-13", "release_year": 2020},
    {"listed_in": "Crime TV Shows, Docuseries", "rating": "TV-MA", "release_year": 2021},
    {"listed_in": "Children & Family Movies, Comedies", "rating": "TV-Y", "release_year": 2019},
]
for item in test_cases:
    res = rf_model.predict(item)
    print(f"{item['listed_in']} ({item['rating']}) -> {res['prediction']} ({res['confidence']:.1%})")

Documentaries, International Movies (PG-13) -> Movie (82.5%)
Crime TV Shows, Docuseries (TV-MA) -> TV Show (97.0%)
Children & Family Movies, Comedies (TV-Y) -> TV Show (67.5%)


## 5. Takeaways

- With leakage removed, the remaining signal is the rating system (TV-style vs MPAA ratings), release year,
  and genres that only really exist in one format (e.g. Reality, Stand-Up & Talk).
- The resulting accuracy is an honest estimate of how well format can be inferred from descriptive metadata.